In [4]:
from datasets import load_dataset
import pandas as pd

print("Downloading dataset...")
dataset = load_dataset("stanfordnlp/sst2")

df_train = pd.DataFrame(dataset['train'])
df_val = pd.DataFrame(dataset['validation'])

print(f"Train size: {len(df_train)}")
print(f"Val size: {len(df_val)}")
print(f"\nColumns: {df_train.columns.tolist()}")
print(f"\nSample:")
print(df_train.head())
print(f"\nLabel distribution:")
print(df_train['label'].value_counts())

Generating test split: 100%|█████████████████████████████████████████████| 1821/1821 [00:00<00:00, 507126.19 examples/s]


Train size: 67349
Val size: 872

Columns: ['idx', 'sentence', 'label']

Sample:
   idx                                           sentence  label
0    0       hide new secretions from the parental units       0
1    1               contains no wit , only labored gags       0
2    2  that loves its characters and communicates som...      1
3    3  remains utterly satisfied to remain the same t...      0
4    4  on the worst revenge-of-the-nerds clichés the ...      0

Label distribution:
label
1    37569
0    29780
Name: count, dtype: int64


In [6]:
import os

# Create folders if they don't exist
os.makedirs("../data/processed", exist_ok=True)

# Save to disk
df_train.to_csv("../data/processed/train.csv", index=False)
df_val.to_csv("../data/processed/val.csv", index=False)
print("Saved train + val to data/processed/")

# Quick exploration
print(f"\nClass balance: {df_train['label'].value_counts(normalize=True).round(3).to_dict()}")
print(f"Avg sentence length: {df_train['sentence'].str.split().str.len().mean():.1f} words")
print(f"Max sentence length: {df_train['sentence'].str.split().str.len().max()} words")

# Sample from each class
print("\n--- NEGATIVE examples ---")
print(df_train[df_train['label']==0]['sentence'].sample(3, random_state=42).tolist())

print("\n--- POSITIVE examples ---")
print(df_train[df_train['label']==1]['sentence'].sample(3, random_state=42).tolist())

Saved train + val to data/processed/

Class balance: {1: 0.558, 0: 0.442}
Avg sentence length: 9.4 words
Max sentence length: 52 words

--- NEGATIVE examples ---
['a dull , dumb and derivative horror ', "if george romero had directed this movie , it would n't have taken the protagonists a full hour to determine that in order to kill a zombie you must shoot it in the head ", 'the acting is amateurish , the cinematography is atrocious ']

--- POSITIVE examples ---
['acted meditation on both the profoundly devastating events of one year ago and the slow , painful healing process that has followed in their wake ', "this odd , poetic road movie , spiked by jolts of pop music , pretty much takes place in morton 's ever-watchful gaze -- and it 's a tribute to the actress , and to her inventive director , that the journey is such a mesmerizing one . ", "directed with purpose and finesse by england 's roger mitchell , who handily makes the move from pleasing , relatively lightweight commercial 

In [8]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib
import os

os.makedirs("../models", exist_ok=True)

# Load data
train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/val.csv")

# TF-IDF features
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train = vectorizer.fit_transform(train['sentence'])
X_val = vectorizer.transform(val['sentence'])

# Train
model = LogisticRegression(max_iter=1000)
model.fit(X_train, train['label'])

# Evaluate
preds = model.predict(X_val)
print(f"Validation Accuracy: {accuracy_score(val['label'], preds):.4f}")
print(classification_report(val['label'], preds, target_names=['negative', 'positive']))

# Save
joblib.dump(model, "../models/model_a_logreg.pkl")
joblib.dump(vectorizer, "../models/vectorizer.pkl")
print("\nModel A saved!")

Validation Accuracy: 0.8142
              precision    recall  f1-score   support

    negative       0.83      0.78      0.80       428
    positive       0.80      0.85      0.82       444

    accuracy                           0.81       872
   macro avg       0.82      0.81      0.81       872
weighted avg       0.82      0.81      0.81       872


Model A saved!


In [10]:
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib
import os

# Absolute paths
base = os.path.expanduser("~/ml-pipeline-monitor")
os.makedirs(f"{base}/data/processed", exist_ok=True)
os.makedirs(f"{base}/models", exist_ok=True)

# Load + save data
dataset = load_dataset("stanfordnlp/sst2")
train = pd.DataFrame(dataset['train'])
val = pd.DataFrame(dataset['validation'])
train.to_csv(f"{base}/data/processed/train.csv", index=False)
val.to_csv(f"{base}/data/processed/val.csv", index=False)
print("Data saved!")

# Train Model A
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train = vectorizer.fit_transform(train['sentence'])
X_val = vectorizer.transform(val['sentence'])

model = LogisticRegression(max_iter=1000)
model.fit(X_train, train['label'])

preds = model.predict(X_val)
print(f"Validation Accuracy: {accuracy_score(val['label'], preds):.4f}")

joblib.dump(model, f"{base}/models/model_a_logreg.pkl")
joblib.dump(vectorizer, f"{base}/models/vectorizer.pkl")
print("Model A saved!")

Data saved!
Validation Accuracy: 0.8142
Model A saved!


In [11]:
import json, os

base = os.path.expanduser("~/ml-pipeline-monitor")
os.makedirs(f"{base}/metrics", exist_ok=True)

metrics_a = {
    "model": "logistic_regression",
    "accuracy": 0.8142,
    "precision_negative": 0.83,
    "recall_negative": 0.78,
    "f1_negative": 0.80,
    "precision_positive": 0.80,
    "recall_positive": 0.85,
    "f1_positive": 0.82,
    "dataset": "stanfordnlp/sst2",
    "features": "tfidf_10000_ngram12"
}

with open(f"{base}/metrics/model_a_metrics.json", "w") as f:
    json.dump(metrics_a, f, indent=2)
print("Done!")

Done!
